# 🎙️ Meeting Minutes Creator V2

This improved version incorporates community best practices:
- **Streaming Output**: See the summary as it's being generated.
- **Local LLM Support**: Support for Ollama (local models) alongside OpenAI.
- **Enhanced UI**: Tabs for cleaner organization and Markdown support.
- **Progress Tracking**: Real-time status updates during transcription.

In [ ]:
import os
from openai import OpenAI
import gradio as gr
from dotenv import load_dotenv
import time

load_dotenv(override=True)

def get_client(provider="OpenAI"):
    if provider == "OpenAI":
        return OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    else: # Ollama / Local
        return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [ ]:
def transcribe_audio(audio_path, provider="OpenAI"):
    """Transcribes audio using Whisper (Cloud or Local if available)"""
    if not audio_path:
        return ""
    
    client = get_client("OpenAI") # Whisper usually cloud-based in this setup
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1", 
            file=audio_file
        )
    return transcript.text

def summarize_meeting_stream(transcript, provider="OpenAI", model_name="gpt-4o-mini"):
    """Summarizes transcript with streaming output"""
    if not transcript:
        yield "No transcript available."
        return
        
    client = get_client(provider)
    
    stream = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "You are a professional secretary. Summarize the following meeting transcript into clear minutes using Markdown formatting. Include key points, decisions, and action items with checkboxes."},
            {"role": "user", "content": transcript}
        ],
        stream=True
    )
    
    partial_message = ""
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            partial_message += chunk.choices[0].delta.content
            yield partial_message

In [ ]:
def process_with_streaming(audio_file, provider, model):
    # Step 1: Transcription
    yield "Transcribing audio...", ""
    transcript = transcribe_audio(audio_file)
    
    # Step 2: Summarization (Streaming)
    yield transcript, "Generating summary..."
    for summary in summarize_meeting_stream(transcript, provider, model):
        yield transcript, summary

with gr.Blocks(theme="base") as demo:
    gr.Markdown("# 🎙️ Meeting Minutes Creator V2")
    
    with gr.Row():
        with gr.Column(scale=1):
            audio_input = gr.Audio(type="filepath", label="Upload Meeting Audio")
            provider_select = gr.Radio(["OpenAI", "Local (Ollama)"], label="LLM Provider", value="OpenAI")
            model_input = gr.Textbox(label="Model Name", value="gpt-4o-mini", placeholder="e.g., gpt-4o-mini or llama3")
            submit_btn = gr.Button("Generate Minutes", variant="primary")
            
        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem("Transcript"):
                    transcript_output = gr.Textbox(label="Full Transcript", lines=15)
                with gr.TabItem("Meeting Minutes"):
                    summary_output = gr.Markdown(label="Minutes")
    
    submit_btn.click(
        process_with_streaming, 
        inputs=[audio_input, provider_select, model_input], 
        outputs=[transcript_output, summary_output]
    )

demo.launch()